In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

import numpy as np
from openai import OpenAI

FEATHERLESS_API_KEY = os.getenv("FEATHERLESS_API_TOKEN")
FEATHERLESS_MODEL = os.getenv("FEATHERLESS_MODEL")

client = OpenAI(
    api_key=FEATHERLESS_API_KEY,
    base_url="https://api.featherless.ai/v1",
)

def generate(prompt: str, system: str = "You are a helpful assistant.", **kwargs) -> str:
    response = client.chat.completions.create(
        model=FEATHERLESS_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        **kwargs,
    )
    return response.choices[0].message.content


In [10]:
system_prompt = """You are a creative fiction writer. Write vivid, engaging prose with strong sensory detail."""

user_prompt = """Write a short opening paragraph for a story about a lighthouse keeper who discovers a mysterious object washed ashore during a storm."""

response = generate(
    prompt=user_prompt,
    system=system_prompt,
    max_tokens=300,
    temperature=0.9,
)

print(f"Model: {FEATHERLESS_MODEL}")
print(f"Response length: {len(response)} characters\n")
print("--- Generated Text ---")
print(response)


Model: NousResearch/Hermes-4-14B
Response length: 557 characters

--- Generated Text ---
The howling wind and relentless waves crashed against the weathered rocks, as if the sea itself was trying to tear the ancient stone lighthouse from its foundation. Keeper Elara shivered, her breath misting in the frigid air, as she scanned the treacherous coastline for any sign of danger. Suddenly, a flicker of something unusual caught her eye – a glint of metal, half-submerged in the churning waters. As the storm intensified, the object began to wash further up the beach, revealing itself to be a strange, ornate compass unlike any she had ever seen.


In [11]:
import json
config = json.load(open("vector_space_config.json"))

In [12]:
from itertools import product, combinations

def setup_states_steps(config):
    dims = config.get('aliases', [str(i) for i in range(config['n_dim'])])
    values = ['', ' reversed'] if config.get('negative_polarity', False) else ['']
    n_steps = config['n_steps']
    step_aliases = config.get('step_aliases',[str(i) for i in range(n_steps)])
    #steps = config.get('step_aliases', [str(i) for i in range(config['n_steps'])])
    possible_states = [i[0]+i[1] for i in list(product(dims, values))]
    return possible_states, step_aliases

def setup_all_possibilities(config):
    dims = config.get('aliases', [str(i) for i in range(config['n_dim'])])
    values = ['', ' reversed'] if config.get('negative_polarity', False) else ['']
    n_steps = config['n_steps']
    step_aliases = {i: config.get('step_aliases',str(i)) for i in range(n_steps)}
    #steps = config.get('step_aliases', [str(i) for i in range(config['n_steps'])])
    possible_states = [i[0]+i[1] for i in list(product(dims, values))]
    print(len(possible_states))
    combos = combinations(possible_states, n_steps)
    return combos, step_aliases

possible_states, step_aliases = setup_states_steps(config)
print(f"Possible states: {possible_states}")
print(f"Step aliases: {step_aliases}")


Possible states: ['Ace of Cups', 'Ace of Cups reversed', 'Ace of Pentacles', 'Ace of Pentacles reversed', 'Ace of Swords', 'Ace of Swords reversed', 'Ace of Wands', 'Ace of Wands reversed', 'Death', 'Death reversed', 'Eight of Cups', 'Eight of Cups reversed', 'Eight of Pentacles', 'Eight of Pentacles reversed', 'Eight of Swords', 'Eight of Swords reversed', 'Eight of Wands', 'Eight of Wands reversed', 'Five of Cups', 'Five of Cups reversed', 'Five of Pentacles', 'Five of Pentacles reversed', 'Five of Swords', 'Five of Swords reversed', 'Five of Wands', 'Five of Wands reversed', 'Four of Cups', 'Four of Cups reversed', 'Four of Pentacles', 'Four of Pentacles reversed', 'Four of Swords', 'Four of Swords reversed', 'Four of Wands', 'Four of Wands reversed', 'Judgement', 'Judgement reversed', 'Justice', 'Justice reversed', 'King of Cups', 'King of Cups reversed', 'King of Pentacles', 'King of Pentacles reversed', 'King of Swords', 'King of Swords reversed', 'King of Wands', 'King of Wands 

In [13]:
from coordinate_systems.state_instantiation import Instantiation

TEST_STATE_STEP_DICT = {'Outcome':'Death'}
instant = Instantiation(possible_states, step_aliases, state_step_dict=TEST_STATE_STEP_DICT)
print(instant.instant)



{'Outcome': 'Death', 'Present': 'Ten of Swords', 'Challenge': 'Six of Cups', 'Conscious Goal': 'Four of Wands', 'Root Cause/Foundation': 'Judgement reversed', 'Recent Past': 'Five of Swords reversed', 'Near Future': 'Death reversed', 'Your Attitude': 'Judgement reversed', 'External Influences': 'The Hierophant', 'Hopes and Fears': 'Page of Pentacles reversed'}


In [14]:
from pathlib import Path

def generate_samples(n, possible_states, step_aliases, state_step_dict=None, write_to_file=None,
                     samples_in_memory=False):
    """
    Generate n random Celtic Cross readings.

    Args:
        n:               number of samples
        possible_states: list of state strings (e.g. all card + polarity combos)
        step_aliases:    list of position name strings
        state_step_dict: optional {step: state} dict to pin certain positions
        write_to_file:   optional filename stem; writes JSONL to coordinate_systems/<name>.jsonl

    Returns:
        list of instant dicts [{step_alias: state, ...}, ...]
    """
    samples = []
    writer = None

    if not samples_in_memory and write_to_file is None:
        print("Samples in memory wasn't requested and no file to write specified... I don't know what to do!")
        return

    if write_to_file is not None:
        out_dir = Path("coordinate_systems")
        out_dir.mkdir(exist_ok=True)
        writer = open(out_dir / f"{write_to_file}.jsonl", "w", encoding="utf-8")

    try:
        for _ in range(n):
            instant = Instantiation(possible_states, step_aliases, state_step_dict).instant
            samples.append(instant)
            if writer is not None:
                writer.write(json.dumps(instant) + "\n")
    finally:
        if writer is not None:
            writer.close()

    return samples


In [15]:
def celtic_cross_prompt(instant: dict) -> str:
    return f"""You're a tarot reader. Can you read my Celtic cross tarot? Give an overview of the reading after, any thoughts you have. I'll draw the cards:

Present: {instant['Present']}
Challenge: {instant['Challenge']}
Conscious Goal: {instant['Conscious Goal']}
Root Cause/Foundation: {instant['Root Cause/Foundation']}
Recent Past: {instant['Recent Past']}
Near Future: {instant['Near Future']}
Your Attitude: {instant['Your Attitude']}
External Influences: {instant['External Influences']}
Hopes and Fears: {instant['Hopes and Fears']}
Outcome: {instant['Outcome']}"""


In [16]:
instant = Instantiation(possible_states, step_aliases).instant
prompt = celtic_cross_prompt(instant)
reading = generate(prompt, max_tokens=4800)
print(reading)


Here's the overview of your Celtic Cross Tarot reading:

Present: Ten of Swords - You are currently facing a difficult situation, feeling overwhelmed and possibly experiencing a sense of defeat or betrayal. It's crucial to acknowledge these emotions and work on healing.

Challenge: The High Priestess reversed - This card suggests that you may be struggling to access your intuition or spiritual guidance, possibly due to distractions or a lack of focus. Try to create a quiet space for self-reflection and reconnect with your inner wisdom.

Conscious Goal: The Moon reversed - Your conscious goal may be unclear or confusing at the moment. You might be experiencing a lack of direction or feeling lost in your endeavors. Take time to clarify your intentions and set realistic goals.

Root Cause/Foundation: Five of Pentacles reversed - This card indicates that you may be facing financial or material challenges. You might feel insecure or inadequate in some areas of your life, which can contribut

In [17]:
print(instant)

{'Present': 'Ten of Swords', 'Challenge': 'The High Priestess reversed', 'Conscious Goal': 'The Moon reversed', 'Root Cause/Foundation': 'Five of Pentacles reversed', 'Recent Past': 'The Hierophant', 'Near Future': 'King of Pentacles reversed', 'Your Attitude': 'Knight of Pentacles', 'External Influences': 'The Hierophant', 'Hopes and Fears': 'Five of Wands', 'Outcome': 'Knight of Pentacles reversed'}


In [18]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

def generate_sample(i):
    instant = Instantiation(possible_states, step_aliases).instant
    prompt = celtic_cross_prompt(instant)
    reading = generate(prompt, max_tokens=4800)
    return i, instant, prompt, reading

n_samples = 10
samples = defaultdict(dict)

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(generate_sample, i): i for i in range(n_samples)}
    for future in as_completed(futures):
        i, instant, prompt, reading = future.result()
        print(i)
        samples[i]['instant'] = instant
        samples[i]['prompt'] = prompt
        samples[i]['reading'] = reading
        print(f"Completed {len(samples)}/{n_samples}")


1
Completed 1/10
0
Completed 2/10
2
Completed 3/10
3
Completed 4/10
4
Completed 5/10
5
Completed 6/10
6
Completed 7/10
7
Completed 8/10
8
Completed 9/10
9
Completed 10/10


In [19]:
len(samples)

10

In [20]:
samples[1]

{'instant': {'Present': 'The Hermit',
  'Challenge': 'Four of Wands',
  'Conscious Goal': 'The Hermit',
  'Root Cause/Foundation': 'Ten of Swords reversed',
  'Recent Past': 'Five of Swords reversed',
  'Near Future': 'Knight of Swords reversed',
  'Your Attitude': 'Eight of Cups',
  'External Influences': 'The Sun',
  'Hopes and Fears': 'Nine of Wands',
  'Outcome': 'Six of Wands reversed'},
 'prompt': "You're a tarot reader. Can you read my Celtic cross tarot? Give an overview of the reading after, any thoughts you have. I'll draw the cards:\n\nPresent: The Hermit\nChallenge: Four of Wands\nConscious Goal: The Hermit\nRoot Cause/Foundation: Ten of Swords reversed\nRecent Past: Five of Swords reversed\nNear Future: Knight of Swords reversed\nYour Attitude: Eight of Cups\nExternal Influences: The Sun\nHopes and Fears: Nine of Wands\nOutcome: Six of Wands reversed",
 'reading': "Here is an overview of your Celtic cross tarot reading:\n\nPresent: The Hermit - You are currently in a phase

In [21]:
import re

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
EMBEDDING_MODEL = EMBEDDING_MODEL or "Qwen/Qwen3-Embedding-8B"

def split_into_chunks(text: str) -> list[str]:
    chunks = re.split(r'(?<=[.!?])\s+|\n+', text)
    return [c.strip() for c in chunks if c.strip()]

def embed(text: str) -> list[float]:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
    )
    return response.data[0].embedding

def embed_samples(samples: dict, max_workers=2) -> dict:
    def embed_one(i):
        chunks = split_into_chunks(samples[i]['reading'])
        embeddings = [embed(chunk) for chunk in chunks]
        return i, chunks, embeddings

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(embed_one, i): i for i in samples}
        for future in as_completed(futures):
            i, chunks, embeddings = future.result()
            samples[i]['chunks'] = chunks
            samples[i]['embeddings'] = embeddings
            print(f"Embedded {sum('embeddings' in v for v in samples.values())}/{len(samples)}")

    return samples



In [22]:
samples = embed_samples(samples)

Embedded 1/10
Embedded 2/10
Embedded 3/10
Embedded 4/10
Embedded 5/10
Embedded 6/10
Embedded 7/10
Embedded 8/10
Embedded 9/10
Embedded 10/10


In [23]:
np.linalg.norm(np.array(samples[0]['embeddings']),axis=1)

array([1.00000005, 1.00000001, 0.99999997, 0.99999998, 1.        ,
       1.00000002, 1.00000005, 0.99999999, 1.00000003, 1.00000002,
       1.00000007, 0.99999994, 0.99999999, 1.00000009, 1.00000001,
       1.00000003, 0.99999998, 0.99999999, 1.        , 1.00000003,
       1.        , 1.        , 1.00000003, 1.00000006, 0.99999998,
       1.00000001, 1.00000006, 0.99999993, 1.00000001, 1.00000004,
       0.99999999, 0.99999995, 1.00000002, 0.99999996, 0.99999999])

In [26]:
from dynamicalembeddings.dynamicalembedding import DynamicalEmbedding
from text_tokenizers.whitespace import WhitespaceTokenizer

In [17]:
entity_check = generate("Create an exhaustive list of entities from the following text... just return a each entity in double quotations, separated by a comma. Here's the text: "+samples[0]['reading'], max_tokens=4800)

In [19]:
entity_check

'"Two of Cups reversed", "Seven of Cups reversed", "Eight of Wands reversed", "The World", "Three of Pentacles", "The Magician", "Three of Pentacles", "Page of Cups reversed", "Seven of Wands reversed", "The Hanged Man", "Celtic Cross spread"'

In [27]:
from collections import defaultdict
d = DynamicalEmbedding(samples[0]['reading'])
d.process_text(tokenizer=WhitespaceTokenizer())
print(d.outputs['tokens'])

[Token(text='Thank', start=0, end=5, id=None, label=None), Token(text='you', start=6, end=9, id=None, label=None), Token(text='for', start=10, end=13, id=None, label=None), Token(text='sharing', start=14, end=21, id=None, label=None), Token(text='your', start=22, end=26, id=None, label=None), Token(text='Celtic', start=27, end=33, id=None, label=None), Token(text='Cross', start=34, end=39, id=None, label=None), Token(text='tarot', start=40, end=45, id=None, label=None), Token(text='reading.', start=46, end=54, id=None, label=None), Token(text='I', start=55, end=56, id=None, label=None), Token(text='will', start=57, end=61, id=None, label=None), Token(text='provide', start=62, end=69, id=None, label=None), Token(text='you', start=70, end=73, id=None, label=None), Token(text='with', start=74, end=78, id=None, label=None), Token(text='an', start=79, end=81, id=None, label=None), Token(text='overview', start=82, end=90, id=None, label=None), Token(text='of', start=91, end=93, id=None, labe

In [20]:
pl_check = generate("Create an exhaustive list of predicate logic statements from the following text.  "+samples[0]['reading'], max_tokens=4800)
print(pl_check)

1. Present: Two of Cups reversed (¬EmotionalConnection ∧ ¬BalancedRelationships)
2. Challenge: Seven of Cups reversed (¬DecisionMaking ∧ ¬OverwhelmedByChoices)
3. Conscious Goal: Eight of Wands reversed (¬Frustration ∧ ¬BlockedEfforts)
4. Root Cause/Foundation: The World (Completion ∧ Fulfillment ∧ Success ∧ SolidFoundation ∧ PotentialForGrowth)
5. Recent Past: Three of Pentacles (WorkingOnProject ∧ Collaboration ∧ WellReceivedEfforts ∧ Progress)
6. Near Future: The Magician (Power ∧ Resources ∧ ManifestDesires ∧ TakeControl ∧ CreatePositiveChange)
7. Your Attitude: Three of Pentacles (ConfidenceInAbilities ∧ StrongTeamwork ∧ StrongCollaboration)
8. External Influences: Page of Cups reversed (¬MixedMessages ∧ ¬EmotionalTurmoil)
9. Hopes and Fears: Seven of Wands reversed (¬UncertaintyAboutFuture ∧ ¬StrugglingToDefendPosition)
10. Outcome: The Hanged Man (StepBack ∧ ReassessSituation ∧ Patience ∧ TrustInProcess ∧ ThingsWorkOut)

11. (¬EmotionalConnection ∧ ¬BalancedRelationships) → Chal

In [21]:
ner_coref_prompt = """
You are an information extraction system. Given the text below, identify all entity mentions and group them into coreference clusters.

Instructions:
1. Find every mention that refers to a real-world entity — this includes proper nouns (e.g., "John Smith"), definite noun phrases (e.g., "the patient", "the company"), and pronouns (e.g., "he", "it", "they").
2. Assign each mention a character start/end offset within the text.
3. Group mentions that refer to the same real-world entity into a single cluster.
4. For each cluster, assign an entity type from this list: PERSON, ORGANIZATION, LOCATION, ROLE, DATE, OTHER.
5. Within each cluster, pick the most specific/informative mention as the "canonical_name".
6. Return ONLY valid JSON, no preamble, no markdown code fences, matching this schema:

{{
  "clusters": [
    {{
      "cluster_id": 0,
      "entity_type": "PERSON",
      "canonical_name": "John Smith",
      "mentions": [
        {{"text": "John Smith", "start": 0, "end": 10}}
      ]
    }}
  ]
}}

Text:
<<<START_TEXT>>>
{text}
<<<END_TEXT>>>
"""

prompt = ner_coref_prompt.format(text=samples[0]['reading'])



In [27]:
ner_coref_check = generate(prompt, max_tokens=4800)

In [24]:
print(ner_coref_check)

{
  "clusters": [
    {
      "cluster_id": 0,
      "entity_type": "OTHER",
      "canonical_name": "Celtic Cross spread",
      "mentions": [
        {"text": "Celtic Cross spread", "start": 36, "end": 51}
      ]
    },
    {
      "cluster_id": 1,
      "entity_type": "OTHER",
      "canonical_name": "Two of Cups reversed",
      "mentions": [
        {"text": "Two of Cups reversed", "start": 71, "end": 87}
      ]
    },
    {
      "cluster_id": 2,
      "entity_type": "OTHER",
      "canonical_name": "Seven of Cups reversed",
      "mentions": [
        {"text": "Seven of Cups reversed", "start": 131, "end": 149}
      ]
    },
    {
      "cluster_id": 3,
      "entity_type": "OTHER",
      "canonical_name": "Eight of Wands reversed",
      "mentions": [
        {"text": "Eight of Wands reversed", "start": 183, "end": 200}
      ]
    },
    {
      "cluster_id": 4,
      "entity_type": "OTHER",
      "canonical_name": "The World",
      "mentions": [
        {"text": "The Worl

In [26]:
prompt = ner_coref_prompt.format(text="Patient came in with a massive headache and conjuctivitis. He said he had been out drinking all night")
ner_coref_check = generate(prompt, max_tokens=4800)
print(ner_coref_check)

{
  "clusters": [
    {
      "cluster_id": 0,
      "entity_type": "PERSON",
      "canonical_name": "Patient",
      "mentions": [
        {"text": "Patient", "start": 0, "end": 7}
      ]
    },
    {
      "cluster_id": 1,
      "entity_type": "PERSON",
      "canonical_name": "He",
      "mentions": [
        {"text": "He", "start": 32, "end": 34}
      ]
    }
  ]
}


In [28]:
text = "Patient came in with a massive headache and conjuctivitis. He said he had been out drinking all night"

# single tok_len
de = DynamicalEmbedding(text, spacing=10)
# → outputs['subtexts'] = {"tok_len_10": [...]}

# multiple tok_lens (shorthand list)
de = DynamicalEmbedding(text, spacing=[5, 10, 20])
# → outputs['subtexts'] = {"tok_len_5": [...], "tok_len_10": [...], "tok_len_20": [...]}

# mixed tok_len and char_len via dict
de = DynamicalEmbedding(text, spacing={"tok_len": [5, 10], "char_len": [100, 300]})
print(de.outputs)
# → outputs['subtexts'] = {"tok_len_5": [...], "tok_len_10": [...], "char_len_100": [...], "char_len_300": [...]}

# no subtexts
de = DynamicalEmbedding(text)
# → outputs has no 'subtexts' key

defaultdict(<class 'dict'>, {'tokens': [Token(text='Patient', start=0, end=7, id=None, label=None), Token(text='came', start=8, end=12, id=None, label=None), Token(text='in', start=13, end=15, id=None, label=None), Token(text='with', start=16, end=20, id=None, label=None), Token(text='a', start=21, end=22, id=None, label=None), Token(text='massive', start=23, end=30, id=None, label=None), Token(text='headache', start=31, end=39, id=None, label=None), Token(text='and', start=40, end=43, id=None, label=None), Token(text='conjuctivitis.', start=44, end=58, id=None, label=None), Token(text='He', start=59, end=61, id=None, label=None), Token(text='said', start=62, end=66, id=None, label=None), Token(text='he', start=67, end=69, id=None, label=None), Token(text='had', start=70, end=73, id=None, label=None), Token(text='been', start=74, end=78, id=None, label=None), Token(text='out', start=79, end=82, id=None, label=None), Token(text='drinking', start=83, end=91, id=None, label=None), Token(t

In [23]:
from text_tokenizers import SpacyTokenizer
import spacy
nlp = spacy.load("en_core_web_sm")
check = nlp("Hi is this working? I'm on an American Airlines flight")

In [32]:
check.doc

Hi is this working? I'm on an American Airlines flight